# Fase 3 · Limpieza de texto y comparación de modelos

Repetimos el ejercicio de clasificación de opiniones, esta vez estudiando si la limpieza, las stopwords y los grupos de dos palabras ayudan al modelo.

**Objetivo:** comparar cambios con datos, sin dar por hecho que limpiar más siempre mejora las predicciones. Usamos el mismo CSV de 60 opiniones y conservamos la separación de la fase 2.

Ejecuta las celdas en orden. Este notebook funciona por sí solo, sin ejecutar las fases anteriores.

## Antes de empezar

Necesitas `dataset_opiniones_60.csv` en la carpeta donde se ejecuta Python. Si utilizas Colab, el CSV debe estar subido al entorno de Colab; tenerlo en la carpeta local de VS Code no lo copia al servidor. No necesitas crear un entorno virtual para usar Colab.

La celda de lectura indicará dónde está buscando si falta el archivo. No copiaremos las opiniones a mano ni descargaremos diccionarios adicionales.

In [1]:
# Solo si faltan las dependencias, descomenta esta línea.
# %pip install pandas scikit-learn

## 1. Importamos las herramientas y cargamos el CSV

`re` permite sustituir patrones de texto. `unicodedata` ayuda a normalizar caracteres Unicode. `clone` crea una copia sin entrenar de un modelo. Las demás herramientas sirven para separar, entrenar y evaluar, como en la fase 2.

In [2]:
import re
import unicodedata
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

ruta_csv = Path("dataset_opiniones_60.csv")
if not ruta_csv.is_file():
    raise FileNotFoundError(
        f"No se encuentra {ruta_csv.name} en {Path.cwd()}. "
        "Sube el CSV a este entorno o modifica ruta_csv con su ubicación en este entorno."
    )

df = pd.read_csv(ruta_csv, encoding="utf-8-sig")
display(df.head())
print(f"Opiniones cargadas: {len(df)}")

,id,comentario,sentimiento
0,1,La experiencia ha sido muy positiva.,positivo
1,2,Me atendieron con poca paciencia y sin amabili...,negativo
2,3,Recibí exactamente lo que había pedido.,positivo
3,4,Funciona de maravilla y cumple lo que promete.,positivo
4,5,El pedido llegó antes de lo esperado.,positivo


Opiniones cargadas: 60


## 2. Comprobamos los datos

Revisamos nulos, textos vacíos y etiquetas. También comprobamos repeticiones después de normalizar mayúsculas y espacios. Si hay duplicados, nos detenemos para revisarlos: no los borramos automáticamente, pues podrían tener etiquetas contradictorias.

La columna `id` es un identificador y no se utiliza como información para predecir.

In [3]:
display(df.isna().sum().to_frame("valores_vacios"))
display(df["sentimiento"].value_counts().to_frame("opiniones"))

assert not df[["comentario", "sentimiento"]].isna().any().any(), "Hay valores vacíos."
assert df["comentario"].map(lambda texto: isinstance(texto, str)).all(), "Hay comentarios que no son texto."
assert df["comentario"].str.strip().ne("").all(), "Hay comentarios sin texto."
assert set(df["sentimiento"]) == {"positivo", "negativo"}, "Revisa las etiquetas."

textos_revisados = df["comentario"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print("Comentarios repetidos al normalizar espacios y mayúsculas:", textos_revisados.duplicated().sum())
assert not textos_revisados.duplicated().any(), "Revisa los comentarios repetidos antes de separar."


,valores_vacios
id,0
comentario,0
sentimiento,0


,opiniones
sentimiento,
positivo,30
negativo,30


Comentarios repetidos al normalizar espacios y mayúsculas: 0


## 3. Diseñamos una limpieza prudente

Nuestra función:

1. Normaliza Unicode para representar de forma coherente los caracteres equivalentes.
2. Convierte a minúsculas.
3. Sustituye enlaces y menciones por espacios.
4. Sustituye números, puntuación y otros signos por espacios, conservando letras Unicode, tildes y `ñ`.
5. Reduce los espacios repetidos y quita los de los extremos.

Sustituir por espacios evita unir palabras: «calidad-precio» pasa a «calidad precio».

Esta función es una decisión para el ejercicio, no una regla universal: los emojis y la puntuación pueden aportar sentimiento y aquí se pierden. En opiniones reales convendría estudiar si conservarlos. El TF-IDF original ya ignora gran parte de la puntuación, por lo que esta limpieza puede cambiar poco este CSV.

In [4]:
def limpiar_texto(texto):
    # NFC conserva las tildes y unifica representaciones equivalentes.
    texto = unicodedata.normalize("NFC", texto).lower()

    # Sustituimos enlaces y menciones por espacios.
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"(?<!\w)@\w+", " ", texto)

    # isalpha() conserva letras Unicode, incluidas ñ, ç y vocales con tilde.
    texto = "".join(caracter if caracter.isalpha() else " " for caracter in texto)

    # Unimos las palabras con un único espacio.
    return " ".join(texto.split())

print(limpiar_texto("¡NO funciona!!! La calidad-precio es pésima. 123"))

no funciona la calidad precio es pésima


## 4. Preparamos las stopwords en español

Las stopwords son palabras frecuentes que decidimos omitir, como «el» o «de». **No todas son prescindibles para todas las tareas.** En este ejercicio usamos una lista pequeña, propia y visible; no es un diccionario completo del español.

Conservamos negaciones y palabras que pueden cambiar el sentimiento: `no`, `ni`, `sin`, `nunca`, `jamás`, `nadie`, `nada`, `poco`, `muy` y `pero`. Quitar «no» convertiría «no recomiendo» en «recomiendo».

También conservamos las formas verbales. No aplicamos stemming ni lematización en esta comparación: queremos entender el efecto de cada cambio sin mezclar demasiadas técnicas.

In [5]:
stopwords_es = [
    "a", "al", "ante", "bajo", "con", "contra", "de", "del", "desde",
    "durante", "el", "ella", "ellas", "ellos", "en", "entre", "esa", "esas",
    "ese", "esos", "esta", "estas", "este", "estos", "hacia", "hasta", "la",
    "las", "le", "les", "lo", "los", "me", "mi", "mis", "nos", "para",
    "por", "que", "se", "sobre", "su", "sus", "te", "tu", "tus", "un",
    "una", "unas", "uno", "unos", "y", "yo"
]

# Usamos el mismo formato para las stopwords que para los comentarios.
stopwords_es = sorted({limpiar_texto(palabra) for palabra in stopwords_es})
palabras_a_conservar = {"no", "ni", "sin", "nunca", "jamás", "nadie", "nada", "poco", "muy", "pero"}
assert not palabras_a_conservar.intersection(stopwords_es)
print("Stopwords utilizadas:")
print(", ".join(stopwords_es))

Stopwords utilizadas:
a, al, ante, bajo, con, contra, de, del, desde, durante, el, ella, ellas, ellos, en, entre, esa, esas, ese, esos, esta, estas, este, estos, hacia, hasta, la, las, le, les, lo, los, me, mi, mis, nos, para, por, que, se, sobre, su, sus, te, tu, tus, un, una, unas, uno, unos, y, yo


### Observamos el antes y el después

Esta tabla sirve para entender la transformación; el entrenamiento usará los textos originales y aplicará la limpieza dentro del Pipeline. La eliminación de stopwords real se hace en el vectorizador.

In [6]:
def quitar_stopwords_ejemplo(texto):
    return " ".join(palabra for palabra in limpiar_texto(texto).split() if palabra not in stopwords_es)

muestra = df[["comentario"]].head(6).copy()
muestra["limpio"] = muestra["comentario"].map(limpiar_texto)
muestra["sin_stopwords"] = muestra["comentario"].map(quitar_stopwords_ejemplo)
display(muestra)

print("Con negación:", quitar_stopwords_ejemplo("No recomiendo esta tienda a nadie"))
print("Sin negación:", quitar_stopwords_ejemplo("Recomiendo esta tienda"))

,comentario,limpio,sin_stopwords
0,La experiencia ha sido muy positiva.,la experiencia ha sido muy positiva,experiencia ha sido muy positiva
1,Me atendieron con poca paciencia y sin amabili...,me atendieron con poca paciencia y sin amabilidad,atendieron poca paciencia sin amabilidad
2,Recibí exactamente lo que había pedido.,recibí exactamente lo que había pedido,recibí exactamente había pedido
3,Funciona de maravilla y cumple lo que promete.,funciona de maravilla y cumple lo que promete,funciona maravilla cumple promete
4,El pedido llegó antes de lo esperado.,el pedido llegó antes de lo esperado,pedido llegó antes esperado
5,Ha sido una compra muy acertada.,ha sido una compra muy acertada,ha sido compra muy acertada


Con negación: no recomiendo tienda nadie
Sin negación: recomiendo tienda


## 5. Reservamos la prueba y preparamos la validación

Mantenemos `test_size=0.2`, `random_state=42` y `stratify=y`, como en la fase 2: 48 opiniones para desarrollar los modelos y 12 para la prueba.

Dentro de las 48 opiniones hacemos **validación cruzada de cinco particiones**: cada modelo aprende con cuatro partes y se valida con la quinta, rotando hasta usar todas. Los modelos se comparan con las mismas particiones.

Elegiremos por el **F1 macro medio**, que da el mismo peso a cada clase. La desviación indica cuánto varía el resultado entre particiones; no es un intervalo de confianza.

**Límite del ejercicio:** ya hemos visto los resultados de estas 12 opiniones en la fase 2. Reutilizarlas permite una comparación didáctica, pero no constituyen una prueba nueva e independiente. Para confirmar mejoras necesitaríamos opiniones nuevas que no hayan guiado nuestras decisiones.

In [7]:
X = df["comentario"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Entrenamiento y validación: {len(X_train)} opiniones")
print(f"Prueba reutilizada de la fase 2: {len(X_test)} opiniones")

Entrenamiento y validación: 48 opiniones
Prueba reutilizada de la fase 2: 12 opiniones


## 6. Comparamos cuatro versiones

| Versión | Cambio respecto a la anterior |
| --- | --- |
| Original | TF-IDF con palabras individuales, como en la fase 2. |
| Limpieza | Añadimos nuestra función de limpieza. |
| Limpieza + stopwords | Añadimos la lista de palabras que omitimos. |
| Limpieza + stopwords + bigramas | Añadimos grupos de dos palabras. |

`ngram_range=(1, 2)` incluye palabras individuales y parejas, por ejemplo «no funciona». Las parejas se forman **después de eliminar stopwords**: «no me gusta» puede producir «no gusta».

El Pipeline recalcula el vocabulario y los pesos TF-IDF dentro de cada partición de entrenamiento, para evitar que la parte de validación intervenga en el aprendizaje. Mantenemos iguales las opciones del clasificador.

In [8]:
vectorizadores = {
    "Original": TfidfVectorizer(),
    "Limpieza": TfidfVectorizer(preprocessor=limpiar_texto),
    "Limpieza + stopwords": TfidfVectorizer(
        preprocessor=limpiar_texto, stop_words=stopwords_es
    ),
    "Limpieza + stopwords + bigramas": TfidfVectorizer(
        preprocessor=limpiar_texto, stop_words=stopwords_es, ngram_range=(1, 2)
    )
}

modelos = {
    nombre: Pipeline([
        ("tfidf", vectorizador),
        ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
    ])
    for nombre, vectorizador in vectorizadores.items()
}

filas_comparacion = []
for nombre, candidato in modelos.items():
    puntuaciones = cross_validate(
        candidato, X_train, y_train, cv=validacion,
        scoring={"exactitud": "accuracy", "f1_macro": "f1_macro"},
        error_score="raise"
    )
    filas_comparacion.append({
        "version": nombre,
        "exactitud_media": puntuaciones["test_exactitud"].mean(),
        "f1_macro_medio": puntuaciones["test_f1_macro"].mean(),
        "desviacion_f1": puntuaciones["test_f1_macro"].std()
    })

comparacion_cv = pd.DataFrame(filas_comparacion).sort_values(
    "f1_macro_medio", ascending=False, kind="stable"
).reset_index(drop=True)
display(comparacion_cv.round(3))

nombre_elegido = comparacion_cv.loc[0, "version"]
print("Versión seleccionada por validación:", nombre_elegido)
print("En caso de empate conservamos la primera versión, que es más sencilla.")

,version,exactitud_media,f1_macro_medio,desviacion_f1
0,Limpieza + stopwords,0.389,0.371,0.153
1,Limpieza + stopwords + bigramas,0.349,0.327,0.156
2,Original,0.344,0.325,0.198
3,Limpieza,0.344,0.325,0.198


Versión seleccionada por validación: Limpieza + stopwords
En caso de empate conservamos la primera versión, que es más sencilla.


## 7. Entrenamos la versión seleccionada y la evaluamos

Ahora entrenamos con las 48 opiniones y aplicamos el modelo a las 12 reservadas. También mostramos el original y una referencia que siempre responde la clase más frecuente.

**La elección ya se ha realizado con la validación cruzada.** No cambiamos de versión para escoger la que más acierte estas 12 opiniones.

In [9]:
modelo_elegido = clone(modelos[nombre_elegido])
modelo_elegido.fit(X_train, y_train)
predicciones = modelo_elegido.predict(X_test)

modelo_original = clone(modelos["Original"])
modelo_original.fit(X_train, y_train)
predicciones_originales = modelo_original.predict(X_test)

referencia = DummyClassifier(strategy="most_frequent")
referencia.fit(X_train.to_frame(), y_train)
predicciones_referencia = referencia.predict(X_test.to_frame())

resumen_prueba = pd.DataFrame([
    {"modelo": nombre, "exactitud": accuracy_score(y_test, pred),
     "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0)}
    for nombre, pred in [
        ("Referencia", predicciones_referencia),
        ("Original", predicciones_originales),
        ("Seleccionado: " + nombre_elegido, predicciones)
    ]
])
display(resumen_prueba.round(3))

resultados = pd.DataFrame({
    "comentario": X_test, "real": y_test,
    "original": predicciones_originales, "seleccionado": predicciones
})
resultados["acierto"] = resultados["real"] == resultados["seleccionado"]
display(resultados)

,modelo,exactitud,f1_macro
0,Referencia,0.500,0.333
1,Original,0.333,0.314
2,Seleccionado: Limpieza + stopwords,0.417,0.413


,comentario,real,original,seleccionado,acierto
27,No recomiendo esta tienda a nadie.,negativo,negativo,negativo,True
58,"El producto es cómodo, práctico y fácil de uti...",positivo,negativo,negativo,False
21,"Me encanta el producto, funciona perfectamente.",positivo,negativo,negativo,False
20,La experiencia ha sido muy negativa.,negativo,positivo,positivo,False
22,El equipo resolvió mi problema enseguida.,positivo,negativo,negativo,False
34,No volvería a comprar en esta tienda.,negativo,negativo,negativo,True
12,Todo el proceso fue sencillo y agradable.,positivo,negativo,positivo,True
24,La solución que me ofrecieron no sirvió de nada.,negativo,negativo,negativo,True
49,El servicio fue rápido y eficiente.,positivo,positivo,positivo,True
44,El servicio fue lento y poco eficiente.,negativo,positivo,positivo,False


### Informe de clasificación y matriz de confusión

`precision` indica cuántas predicciones de una clase son correctas; `recall` indica cuántos casos reales de esa clase hemos detectado. F1 combina ambas medidas. `support` cuenta los casos reales de cada clase.

En la matriz, las filas indican la clase real y las columnas la predicha.

In [10]:
etiquetas = ["negativo", "positivo"]
print(classification_report(y_test, predicciones, labels=etiquetas, zero_division=0))
display(pd.DataFrame(
    confusion_matrix(y_test, predicciones, labels=etiquetas),
    index=["Real negativo", "Real positivo"],
    columns=["Predicho negativo", "Predicho positivo"]
))
print("Opiniones mal clasificadas:")
display(resultados.loc[~resultados["acierto"]])

,Predicho negativo,Predicho positivo
Real negativo,3,3
Real positivo,4,2


,comentario,real,original,seleccionado,acierto
58,"El producto es cómodo, práctico y fácil de uti...",positivo,negativo,negativo,False
21,"Me encanta el producto, funciona perfectamente.",positivo,negativo,negativo,False
20,La experiencia ha sido muy negativa.,negativo,positivo,positivo,False
22,El equipo resolvió mi problema enseguida.,positivo,negativo,negativo,False
44,El servicio fue lento y poco eficiente.,negativo,positivo,positivo,False
56,Estoy muy contenta con mi compra.,positivo,negativo,negativo,False
45,La experiencia de compra ha sido horrible.,negativo,positivo,positivo,False


              precision    recall  f1-score   support

    negativo       0.43      0.50      0.46         6
    positivo       0.40      0.33      0.36         6

    accuracy                           0.42        12
   macro avg       0.41      0.42      0.41        12
weighted avg       0.41      0.42      0.41        12

Opiniones mal clasificadas:


## 8. Probamos tus seis opiniones

Las opiniones se pasan como texto original: el modelo seleccionado realiza las transformaciones correspondientes. Mostramos también la predicción original para observar cambios.

Estas frases son demostraciones, no una evaluación independiente. Algunas expresiones se parecen a las del entrenamiento.

In [11]:
nuevas_opiniones = [
    "Estoy feliz con la compra, todo funciona de maravilla",
    "El paquete llegó destrozado y nadie me ayudó",
    "No estoy contenta con el producto",
    "Me encanta, el producto es maravilloso",
    "El pedido llegó tarde y estoy muy enfadada",
    "La calidad es estupenda y lo recomiendo"
]

display(pd.DataFrame({
    "comentario": nuevas_opiniones,
    "original": modelo_original.predict(nuevas_opiniones),
    "seleccionado": modelo_elegido.predict(nuevas_opiniones)
}))

,comentario,original,seleccionado
0,"Estoy feliz con la compra, todo funciona de ma...",positivo,positivo
1,El paquete llegó destrozado y nadie me ayudó,negativo,negativo
2,No estoy contenta con el producto,negativo,negativo
3,"Me encanta, el producto es maravilloso",negativo,positivo
4,El pedido llegó tarde y estoy muy enfadada,negativo,negativo
5,La calidad es estupenda y lo recomiendo,positivo,positivo


## 9. Sacamos conclusiones a partir de los resultados

La celda siguiente informa del resultado real, aunque la limpieza no mejore el modelo. No cambiamos el azar ni las etiquetas para conseguir una puntuación más alta.

In [12]:
exactitud_original = accuracy_score(y_test, predicciones_originales)
exactitud_elegido = accuracy_score(y_test, predicciones)
print("Versión elegida por F1 macro en validación:", nombre_elegido)
print(f"Exactitud original en prueba: {exactitud_original:.2%}")
print(f"Exactitud del seleccionado en prueba: {exactitud_elegido:.2%}")
if exactitud_elegido > exactitud_original:
    print("El seleccionado obtiene más aciertos que el original en esta pequeña prueba.")
elif exactitud_elegido == exactitud_original:
    print("Ambos tienen la misma exactitud en esta pequeña prueba.")
else:
    print("El seleccionado obtiene menos aciertos que el original en esta pequeña prueba.")
print("Cada acierto de 12 cambia la exactitud unos 8,33 puntos porcentuales.")
print("Hay que confirmar los resultados con más opiniones nuevas y variadas.")

Versión elegida por F1 macro en validación: Limpieza + stopwords
Exactitud original en prueba: 33.33%
Exactitud del seleccionado en prueba: 41.67%
El seleccionado obtiene más aciertos que el original en esta pequeña prueba.
Cada acierto de 12 cambia la exactitud unos 8,33 puntos porcentuales.
Hay que confirmar los resultados con más opiniones nuevas y variadas.


### Qué podemos aprender

- Limpiar el texto no garantiza una mejora: también puede eliminar información útil.
- Eliminar stopwords debe adaptarse a la tarea. Las negaciones son especialmente importantes para clasificar sentimientos.
- Los bigramas captan algunas relaciones entre palabras, pero necesitan suficientes ejemplos para aprenderlas.
- Con 60 opiniones el resultado es inestable y puede variar mucho según la separación.
- Para continuar, conviene recopilar más opiniones reales, revisar sus etiquetas y reservar un conjunto nuevo para una evaluación final.

**Para practicar:** compara la fila «Limpieza» con «Limpieza + stopwords». ¿Mejora el F1 macro medio? Revisa después si añadir bigramas ayuda. Explica los resultados aunque contradigan tu expectativa.

### Documentación de referencia

- [Extracción de características y stopwords en Scikit-learn](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [Validación cruzada en Scikit-learn](https://scikit-learn.org/stable/modules/cross_validation.html)
